<a href="https://colab.research.google.com/github/aalaa340/arabic-LLM-from-scratch/blob/main/Arabic_LLM_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

with open("arabic_dataset-1.txt",'r',encoding='utf-8') as f:
  text=f.read()
print("length of data set in characters :", len(text))
#lets see the first 100 characters on the file
print(text[:100])

length of data set in characters : 1115394
في المكتبة الهادئة، تقلبت صفحات الكتب، وكان لكل قارئ سؤال يبحث له عن
إجابة. القراءة تفتح أمام الإنسا


In [ ]:
#seeing all possiple characters the make a words
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(chars)
print(vocab_size)


['\n', ' ', '.', '،', 'ء', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي']
39


In [ ]:
#simple tokenization

#encoder:take string and output alist of integers
stoi={ch:i for i,ch in enumerate(chars)}
encode=lambda s:[stoi[ch] for ch in s]

#decoder:take a list of integer , output a string
itos={i:ch for i,ch in enumerate(chars)}
decode=lambda l :[itos[i]for i in l]

print(encode("السلام عليكم"))
print(decode(encode("السلام عليكم")))

[9, 32, 21, 32, 9, 33, 1, 27, 32, 38, 31, 33]
['ا', 'ل', 'س', 'ل', 'ا', 'م', ' ', 'ع', 'ل', 'ي', 'ك', 'م']


In [ ]:
# encode the entire text and store it into torch.tensor
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([29, 38,  1,  9, 32, 33, 31, 12, 10, 11,  1,  9, 32, 35,  9, 17,  8, 11,
         3,  1, 12, 30, 32, 10, 12,  1, 23, 29, 15,  9, 12,  1,  9, 32, 31, 12,
        10,  3,  1, 36, 31,  9, 34,  1, 32, 31, 32,  1, 30,  9, 19,  8,  1, 21,
         6,  9, 32,  1, 38, 10, 15, 13,  1, 32, 35,  1, 27, 34,  0,  7, 14,  9,
        10, 11,  2,  1,  9, 32, 30, 19,  9,  4, 11,  1, 12, 29, 12, 15,  1,  5,
        33,  9, 33,  1,  9, 32,  7, 34, 21,  9])


In [ ]:
#split up the data into train and validation sets
n=int(0.8*len(data))
train_data=data[:n]
val_data=data[n:]

now our text is coverted into integers and its divided into train and validation
we want to make a code to explain that when we give a transformer "t" as character it will predict the next of it "h" for example .

In [ ]:
block_size=8
train_data[:block_size + 1] #the first 9 characters in the train_data
#in the 9 character there is 8 examples
# if the context is 1 -->4 ix next
#if the context is 1,4--> 7 is next ...
x=train_data[:block_size]
y=train_data[1:block_size+1]
for i in range(block_size):
  context=x[:i+1]
  target=y[i]
  print(f"when input is {context} the target is {target}")
print(context)

when input is tensor([29]) the target is 38
when input is tensor([29, 38]) the target is 1
when input is tensor([29, 38,  1]) the target is 9
when input is tensor([29, 38,  1,  9]) the target is 32
when input is tensor([29, 38,  1,  9, 32]) the target is 33
when input is tensor([29, 38,  1,  9, 32, 33]) the target is 31
when input is tensor([29, 38,  1,  9, 32, 33, 31]) the target is 12
when input is tensor([29, 38,  1,  9, 32, 33, 31, 12]) the target is 10
tensor([29, 38,  1,  9, 32, 33, 31, 12])


now inspide of making one batch as above , i will make a batch dimention just because efficiency , we will have multiple batches for multiple chuncks of text

In [ ]:
torch.manual_seed(1337)
batch_size=4
block_size=8 #4 batches ,each batch has 8 characters
def get_batch(split):
  data=train_data if split=="train" else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,)) #4 random numbers each one is the startpoint of a sequence
  x=torch.stack([data[i:i+block_size]for i in ix])
  y=torch.stack([data[i+1:i+block_size+1]for i in ix])
  return x,y

xb,yb=get_batch("train")
print("inputs:" , xb.shape)
print(xb)

print("target:" , yb.shape)
print(yb)

print("___________________________")
for b in range(batch_size ):
  for i in range(block_size):
    context=xb[b:i+1]
    target=yb[b,i]
    print(f"when input is {context.tolist()} the target is {target}")




inputs: torch.Size([4, 8])
tensor([[ 9, 34, 12,  1,  9, 32, 22, 36],
        [34, 15,  1,  9, 32, 15, 38,  9],
        [36, 33,  2,  1,  9, 32, 25, 10],
        [ 9, 32,  9, 31, 12, 22,  9, 29]])
target: torch.Size([4, 8])
tensor([[34, 12,  1,  9, 32, 22, 36,  9],
        [15,  1,  9, 32, 15, 38,  9, 11],
        [33,  2,  1,  9, 32, 25, 10, 38],
        [32,  9, 31, 12, 22,  9, 29,  2]])
___________________________
when input is [[9, 34, 12, 1, 9, 32, 22, 36]] the target is 34
when input is [[9, 34, 12, 1, 9, 32, 22, 36], [34, 15, 1, 9, 32, 15, 38, 9]] the target is 12
when input is [[9, 34, 12, 1, 9, 32, 22, 36], [34, 15, 1, 9, 32, 15, 38, 9], [36, 33, 2, 1, 9, 32, 25, 10]] the target is 1
when input is [[9, 34, 12, 1, 9, 32, 22, 36], [34, 15, 1, 9, 32, 15, 38, 9], [36, 33, 2, 1, 9, 32, 25, 10], [9, 32, 9, 31, 12, 22, 9, 29]] the target is 9
when input is [[9, 34, 12, 1, 9, 32, 22, 36], [34, 15, 1, 9, 32, 15, 38, 9], [36, 33, 2, 1, 9, 32, 25, 10], [9, 32, 9, 31, 12, 22, 9, 29]] the t

biagram language model :

In [ ]:
import torch
import torch.nn as nn  # to use neural network component
from torch.nn import functional as f #to use loss function later
torch.manual_seed(1337)

class BiagramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,target=None):
    logits=self.token_embedding_table(idx)
    if target is None:
      loss=None
    else:
      B,T,C=logits.shape
      logits=logits.view(B*T,C)
      targets=target.view(B*T)
      loss=f.cross_entropy(logits , targets)
      return logits , loss

m= BiagramLanguageModel(vocab_size)
out=m(xb,yb)
print(out)

(tensor([[-0.7995,  0.2922,  1.3143,  ..., -0.5512,  1.0477,  1.6187],
        [ 0.4313,  0.3725, -0.3151,  ...,  1.6543, -0.1434,  0.5471],
        [ 0.0750,  0.6550, -0.5006,  ..., -0.0157,  0.2478,  0.5076],
        ...,
        [ 0.6586, -1.3565, -1.8218,  ...,  1.0874, -0.9978,  0.8634],
        [-0.7995,  0.2922,  1.3143,  ..., -0.5512,  1.0477,  1.6187],
        [ 0.1850, -0.8069,  0.1648,  ..., -1.4296,  0.2347, -1.2034]],
       grad_fn=<ViewBackward0>), tensor(3.9196, grad_fn=<NllLossBackward0>))


In [ ]:
#generate new tokens one by one
def generate(self,idx,max_new_tokens):
